# Cross-Model Evaluation and Best-Model Comparison

Evaluation-only Kaggle workflow for MobileNetV2, EfficientNetB0, EfficientNet-Lite0, MobileNetV3-Small, and ShuffleNetV2 0.5x.
Architectures: mobilenetv2, efficientnet_b0, efficientnet_lite0, mobilenetv3_small, shufflenetv2_05.
It compares float and available TFLite artifacts without retraining or overwriting model files.
Set `COPY_DATASET_TO_WORKING = True` in the setup cell to optionally copy the validated benchmark to Kaggle working storage.


In [ ]:
import os

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
REPO_DIR = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
KAGGLE_INPUT_ROOT = '/kaggle/input'
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = '/kaggle/working/leaf_dataset/benchmark'
DATASET_BASE = None
MODEL_ROOT = '/kaggle/working/leaf_models'
OUTPUT_DIR = '/kaggle/working/leaf_model_comparison'
MAX_TEST_SAMPLES = None  # Set to 500 for a fast smoke evaluation.
EVAL_BATCH_SIZE = 32
print('Dataset search root:', KAGGLE_INPUT_ROOT)
print('Models:', MODEL_ROOT)
print('Results:', OUTPUT_DIR)
print("Copy benchmark to Kaggle working storage:", COPY_DATASET_TO_WORKING)


## Environment setup

In [ ]:
import os
import subprocess
import sys

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
required_paths = ['requirements.txt', 'training/train.py', 'training/evaluate.py', 'notebooks/model_comparison_common.py']
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f'Repository clone is incomplete; missing: {missing_paths}')
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'notebooks'))
print('Complete project repository is ready:', REPO_DIR)

## Install project dependencies

In [ ]:
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', requirements_path], check=True)
print('Dependencies installed from:', requirements_path)

## Evaluate all five architectures

In [ ]:
import json
import shutil
from collections import deque
from pathlib import Path

from dataset.taxonomy import CANONICAL_CLASSES
from training.utils import load_benchmark_class_names

_EXPECTED_CLASSES = list(CANONICAL_CLASSES)
_SPLITS = ("train", "val", "test")
_DESCENDABLE_DIRS = {"processed", "dataset", "data"}

def _validate_benchmark_root(dataset_root):
    dataset_root = Path(dataset_root)
    manifest_path = dataset_root / "class_names.json"
    if not manifest_path.is_file():
        raise ValueError(f"Canonical class manifest is missing: {manifest_path}")
    class_names = load_benchmark_class_names(str(dataset_root))
    if class_names != _EXPECTED_CLASSES:
        raise ValueError("Benchmark manifest does not match the canonical class order")
    for split in _SPLITS:
        split_dir = dataset_root / split
        if not split_dir.is_dir():
            raise ValueError(f"Benchmark split is missing: {split_dir}")
        observed = {item.name for item in split_dir.iterdir() if item.is_dir()}
        if observed != set(_EXPECTED_CLASSES):
            raise ValueError(f"{split} folders do not match the canonical 72-class taxonomy")
    return class_names

def _benchmark_mount_roots(input_root):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is missing: {input_root}")
    roots = []
    for item in input_root.iterdir():
        if not item.is_dir():
            continue
        if item.name == "datasets":
            for owner in item.iterdir():
                if owner.is_dir():
                    roots.extend(child for child in owner.iterdir() if child.is_dir())
        else:
            roots.append(item)
    return roots

def find_benchmark_root(input_root=KAGGLE_INPUT_ROOT):
    candidates = []
    visited = set()
    for mount_root in _benchmark_mount_roots(input_root):
        queue = deque([(mount_root, 0)])
        while queue:
            current, depth = queue.popleft()
            current = current.resolve()
            if current in visited:
                continue
            visited.add(current)
            try:
                children = [child for child in current.iterdir() if child.is_dir()]
            except OSError:
                continue
            child_names = {child.name for child in children}
            if set(_SPLITS).issubset(child_names):
                if (current / "class_names.json").is_file():
                    try:
                        _validate_benchmark_root(current)
                    except (OSError, ValueError, json.JSONDecodeError):
                        pass
                    else:
                        candidates.append(current)
                continue
            if depth < 8:
                queue.extend((child, depth + 1) for child in children if child.name.lower() in _DESCENDABLE_DIRS)
    if not candidates:
        raise FileNotFoundError("Could not find a canonical train/val/test benchmark with class_names.json under /kaggle/input")
    deepest = max(len(path.parts) for path in candidates)
    candidates = sorted(path for path in candidates if len(path.parts) == deepest)
    if len(candidates) != 1:
        raise RuntimeError("Multiple canonical benchmark roots found; select one explicitly: " + ", ".join(map(str, candidates)))
    return candidates[0]

DATASET_BASE = str(find_benchmark_root())
_validate_benchmark_root(DATASET_BASE)
print("Validated benchmark:", DATASET_BASE)
if COPY_DATASET_TO_WORKING:
    working_dataset_root = Path(WORKING_DATASET_ROOT)
    if working_dataset_root.exists():
        _validate_benchmark_root(working_dataset_root)
        print("Reusing valid benchmark copy:", working_dataset_root)
    else:
        working_dataset_root.parent.mkdir(parents=True, exist_ok=True)
        print("Copying benchmark to Kaggle working storage...")
        shutil.copytree(DATASET_BASE, working_dataset_root)
        _validate_benchmark_root(working_dataset_root)
        print("Benchmark copy complete:", working_dataset_root)
    DATASET_BASE = str(working_dataset_root)
print("Comparison dataset:", DATASET_BASE)

from model_comparison_common import run_comparison

recommendation, summary_df, ranking_df = run_comparison(
    repo_dir=REPO_DIR,
    dataset_base=DATASET_BASE,
    model_root=MODEL_ROOT,
    output_dir=OUTPUT_DIR,
    max_test_samples=MAX_TEST_SAMPLES,
    eval_batch_size=EVAL_BATCH_SIZE,
)
display(summary_df)
display(ranking_df)

## Inspect the recommendation and plots

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

print('Best float model:', recommendation['best_float_accuracy'])
print('Best edge model:', recommendation['best_edge_deployment'])
print('Best balanced model:', recommendation['best_balanced'])

for plot_name in ('accuracy_f1_comparison.png', 'size_vs_latency.png', 'float_to_int8_drop.png', 'pareto_frontier.png'):
    plot_path = Path(OUTPUT_DIR) / 'plots' / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

report_path = Path(OUTPUT_DIR) / 'best_model_report.md'
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))
print('All outputs:', OUTPUT_DIR)
print('Archive:', f'{OUTPUT_DIR}.zip')